In [3]:
import duckdb
import polars as pl

con = duckdb.connect("../data/m5.db")

with open("../queries/create_base_dataset.sql") as f:
    query = f.read()

df = (
    con.execute(query)
    .pl()
    .with_columns([
        pl.col("sales").cast(pl.Int32),
        pl.col("avg_sell_price").cast(pl.Float32),
        pl.col("snap").cast(pl.Int8),
        pl.col("agg_id").cast(pl.Categorical),
        pl.col("dept_id").cast(pl.Categorical),
        pl.col("cat_id").cast(pl.Categorical),
        pl.col("store_id").cast(pl.Categorical),
        pl.col("state_id").cast(pl.Categorical),
    ])
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [6]:
df

agg_id,dept_id,cat_id,store_id,state_id,date,event_name_1,event_type_1,event_name_2,event_type_2,snap,sales,avg_sell_price
cat,cat,cat,cat,cat,date,str,str,str,str,i8,i32,f32
"""FOODS_1_FOODS_CA_1_CA""","""FOODS_1""","""FOODS""","""CA_1""","""CA""",2011-01-29,null,null,null,null,0,297,2.293535
"""FOODS_1_FOODS_CA_1_CA""","""FOODS_1""","""FOODS""","""CA_1""","""CA""",2011-01-30,null,null,null,null,0,284,2.266514
"""FOODS_1_FOODS_CA_1_CA""","""FOODS_1""","""FOODS""","""CA_1""","""CA""",2011-01-31,null,null,null,null,0,214,2.108131
"""FOODS_1_FOODS_CA_1_CA""","""FOODS_1""","""FOODS""","""CA_1""","""CA""",2011-02-01,null,null,null,null,1,175,2.287943
"""FOODS_1_FOODS_CA_1_CA""","""FOODS_1""","""FOODS""","""CA_1""","""CA""",2011-02-02,null,null,null,null,1,182,2.177802
…,…,…,…,…,…,…,…,…,…,…,…,…
"""HOUSEHOLD_2_HOUSEHOLD_WI_3_WI""","""HOUSEHOLD_2""","""HOUSEHOLD""","""WI_3""","""WI""",2016-05-18,null,null,null,null,0,133,4.065564
"""HOUSEHOLD_2_HOUSEHOLD_WI_3_WI""","""HOUSEHOLD_2""","""HOUSEHOLD""","""WI_3""","""WI""",2016-05-19,null,null,null,null,0,141,4.170355
"""HOUSEHOLD_2_HOUSEHOLD_WI_3_WI""","""HOUSEHOLD_2""","""HOUSEHOLD""","""WI_3""","""WI""",2016-05-20,null,null,null,null,0,230,4.076739


In [ ]:
for col in df.select_dtypes(include=["str", "datetime"]).columns:
    print(df[col].value_counts())
    print("="*30)

In [ ]:
for col in df.select_dtypes(include=["int", "float"]).columns:
    print(df[col].describe())
    print("="*30)

In [ ]:
def plot_agg_id_sales(df, agg_id=None):
    if agg_id is None:
        agg_id = df["agg_id"].sample(1).iloc[0]
    
    df_sample = df[df["agg_id"] == agg_id].copy()
    
    plt.figure(figsize=(15, 6))
    plt.plot(df_sample["date"], df_sample["sales"])
    plt.title(f"Sales over time for agg_id: {agg_id}")
    plt.xlabel("Date")
    plt.ylabel("Sales")
    plt.xticks(rotation=45)
    plt.grid()
    plt.tight_layout()
    plt.show()

In [ ]:
plot_agg_id_sales(df)

In [ ]:
df_statistics = df.groupby("agg_id")["sales"].agg(["mean", "std", "min", "max"]).reset_index()

df_statistics['cv'] = df_statistics['std'] / df_statistics['mean']

print(df_statistics.nsmallest(10, 'cv').to_string(index=False))
print(df_statistics.nlargest(10, 'cv').to_string(index=False))

In [ ]:
plot_agg_id_sales(df, agg_id="HOUSEHOLD_2_HOUSEHOLD_CA_3_CA")
plot_agg_id_sales(df, agg_id="FOODS_2_FOODS_CA_2_CA")

## Réplica en Polars puro (sin DuckDB)

Misma agregación semanal que `queries/create_base_dataset_weekly.sql`, pero resuelta con
`LazyFrame` de Polars: `unpivot` para pasar `sales_train_evaluation.csv` de ancho a largo,
`join` con `calendar` y `sell_prices`, `group_by` + `agg`, y streaming engine para no explotar RAM.

In [ ]:
import polars as pl

RAW_DIR = "../data/raw"
ID_VARS = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]

# 1. Escaneo perezoso de los CSV crudos (nada se carga en RAM todavía)
calendar = pl.scan_csv(f"{RAW_DIR}/calendar.csv", try_parse_dates=True)
sell_prices = pl.scan_csv(f"{RAW_DIR}/sell_prices.csv")
sales_wide = pl.scan_csv(f"{RAW_DIR}/sales_train_evaluation.csv")

day_cols = [c for c in sales_wide.collect_schema().names() if c not in ID_VARS]

# 2. sales_train_evaluation viene en ancho (una columna por día) -> lo pasamos a largo,
#    equivalente al UNPIVOT que hace DuckDB en queries/create_database.sql
sales_long = sales_wide.unpivot(
    on=day_cols,
    index=ID_VARS,
    variable_name="d",
    value_name="sales",
).with_columns(pl.col("sales").cast(pl.Int32))

# 3. Join + agregación semanal (réplica de queries/create_base_dataset_weekly.sql)
weekly = (
    sales_long.join(calendar, on="d", how="left")
    .join(sell_prices, on=["store_id", "item_id", "wm_yr_wk"], how="left")
    .with_columns(
        pl.when(pl.col("state_id") == "CA")
        .then(pl.col("snap_CA"))
        .when(pl.col("state_id") == "TX")
        .then(pl.col("snap_TX"))
        .when(pl.col("state_id") == "WI")
        .then(pl.col("snap_WI"))
        .alias("snap")
    )
    .group_by(ID_VARS + ["wm_yr_wk"])
    .agg(
        pl.col("date").min().alias("week_start_date"),
        pl.col("date").max().alias("week_end_date"),
        pl.col("event_name_1").drop_nulls().n_unique().alias("n_events_1"),
        pl.col("event_name_2").drop_nulls().n_unique().alias("n_events_2"),
        pl.col("snap").sum().alias("snap_days"),
        pl.col("sales").sum().alias("sales"),
        (pl.col("sales") * pl.col("sell_price")).sum().alias("_sales_price_sum"),
    )
    # NULLIF(SUM(sales), 0) del SQL original: evitar división por cero
    .with_columns(
        pl.when(pl.col("sales") == 0)
        .then(None)
        .otherwise(pl.col("_sales_price_sum") / pl.col("sales"))
        .alias("avg_sell_price")
    )
    .drop("_sales_price_sum")
    .sort(["id", "wm_yr_wk"])
    # 4. Optimizar dtypes (crucial para que quepa cómodo en RAM)
    .with_columns(
        pl.col("dept_id").cast(pl.Categorical),
        pl.col("cat_id").cast(pl.Categorical),
        pl.col("store_id").cast(pl.Categorical),
        pl.col("state_id").cast(pl.Categorical),
        pl.col("snap_days").cast(pl.Int16),
        pl.col("sales").cast(pl.Int32),
        pl.col("avg_sell_price").cast(pl.Float32),
    )
)

# 5. Ejecutar con el motor streaming: procesa por bloques sin colapsar RAM
df_pl = weekly.collect(engine="streaming")

print(df_pl.shape)
df_pl.head()